In [61]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph , START ,END 
from pydantic import BaseModel , Field 
from typing  import TypedDict , Annotated , Literal 
from PIL import Image 
import io 



In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model ="llama-3.3-70b-versatile" , temperature=0.3)


In [ ]:
class Structuredoutput(BaseModel):
    risk : Literal["low" , "medium" , "high"] = Field(description="Severity level based on reported physical symptoms.")
    
    

In [20]:
structured_model = model.with_structured_output(Structuredoutput)


In [21]:
class PatientState(TypedDict):
    symptoms : str 
    risk_level : str 
    action_plan : str

In [65]:
def risk_cheaking(state : PatientState ) ->PatientState:
    symptom = state['symptoms']
    prompt = f"After deeply anlyzing the symptoms of patient provide me risk level  low , medium , high \n {symptom}"
    output = structured_model.invoke(prompt)
    return {"risk_level" : output.risk }


In [66]:
def low_risk(state : PatientState) -> PatientState:
    symptoms = state['symptoms']
    prompt = f"You are a nurse. your duty is to analyze the symptoms \n{symptoms} Provide gentle self-care advice and over-the-counter care tips."
    output = model.invoke(prompt).content
    return {'action_plan' : output}


In [67]:
def medium_risk(state : PatientState) -> PatientState:
    symptoms = state['symptoms']
    prompt = f"You are a Senior Doctor. your duty is to analyze the symptoms \n{symptoms} Provide appointement i can take to being normal ."
    output = model.invoke(prompt).content
    return {'action_plan' : output}

In [68]:
def high_risk(state : PatientState) -> PatientState:
    symptoms = state['symptoms']
    prompt = f"You are a Senior Doctor . your duty is to analyze the symptoms \n{symptoms} provide me the emergency advice for high risk"
    output = model.invoke(prompt).content
    return {'action_plan' : output}

In [69]:
def route_risk(state: PatientState) -> str:
    return state["risk_level"]

In [70]:
graph = StateGraph(PatientState)


In [71]:
graph.add_node("risk_cheak" , risk_cheaking)
graph.add_node("low" , low_risk)
graph.add_node("medium" , medium_risk)
graph.add_node("high"  , high_risk)


In [72]:
graph.add_edge(START , "risk_cheak")
graph.add_conditional_edges("risk_cheak" , route_risk , {"low": "low", "medium": "medium", "high": "high"})
graph.add_edge("low", END)
graph.add_edge("medium", END)
graph.add_edge("high", END)

In [73]:
workflow = graph.compile()

In [74]:
png = workflow.get_graph().draw_mermaid_png()
img = Image.open(io.BytesIO(png))
img.show()

In [77]:
in_state = {"symptoms" : "Sudden numbness on the right side of my face, slurred speech, and loss of balance."}
output= workflow.invoke(in_state)
print(output['risk_level'])
print(output['action_plan'])

high
**EMERGENCY SITUATION**

Based on the symptoms you've described - sudden numbness on the right side of your face, slurred speech, and loss of balance - I strongly suspect that you may be experiencing a **STROKE**.

**IMMEDIATE ACTION REQUIRED**

As a high-risk situation, it's crucial to act quickly. Please follow these emergency steps:

1. **CALL FOR EMERGENCY MEDICAL HELP**: Immediately dial the emergency number in your country (e.g., 911 in the US) or have someone else do it for you. Inform the operator that you suspect a stroke.
2. **STAY CALM AND STILL**: Try to remain as calm as possible and avoid moving around, as this can worsen the condition.
3. **NOTE THE TIME**: Make a note of the time when your symptoms started. This information is critical for medical professionals to determine the best course of treatment.
4. **DO NOT DRIVE**: Do not attempt to drive yourself to the hospital, as your condition may worsen, and you may pose a risk to yourself and others on the road.
5. 